# Barebones SpatialData Image Write Test (manual `Image2DModel`, 10k crop)

This notebook does only four things:

1. read the merged OME-TIFF metadata
2. lazily read the full image with `dask_image.imread(...)`
3. crop a 10k x 10k window
4. construct `SpatialData(images={...})`
5. write the image-only SpatialData store and reopen it

This is the manual `Image2DModel` path without `sopa`.

No labels, shapes, segmentation, or tables are included.


In [1]:
from __future__ import annotations

import os
import json
import shutil
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import dask.array as da
import spatialdata as sd
import tifffile
from dask_image.imread import imread
from spatialdata import SpatialData
from spatialdata.models import Image2DModel


In [2]:
OUTPUTS_DIR = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs")
FULL_MERGE_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329_full_merge.ome.tif")
CHANNEL_MAP_PATH = OUTPUTS_DIR / "channel_map.generated.json"
IMAGE_LAYER = "full_image"
CHUNK_SHAPE = (1, 512, 512)
CROP_X = 0
CROP_Y = 0
CROP_WIDTH = 10_000
CROP_HEIGHT = 10_000
SCALE_FACTORS = [2, 2, 2, 2]
WRITE_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_lazy_no_sopa_SLIDE-0329_crop_10000.sdata.zarr")

assert FULL_MERGE_PATH.exists(), FULL_MERGE_PATH
assert CHANNEL_MAP_PATH.exists(), CHANNEL_MAP_PATH

print({
    "spatialdata": sd.__version__,
    "full_merge_path": str(FULL_MERGE_PATH),
    "write_path": str(WRITE_PATH),
    "chunk_shape": CHUNK_SHAPE,
    "crop_box": {"x": CROP_X, "y": CROP_Y, "width": CROP_WIDTH, "height": CROP_HEIGHT},
    "scale_factors": SCALE_FACTORS,
})


{'spatialdata': '0.7.2', 'full_merge_path': '/mnt/c/Analysis/Data_prototype/SLIDE-0329_full_merge.ome.tif', 'write_path': '/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_lazy_no_sopa_SLIDE-0329.sdata.zarr', 'chunk_shape': (1, 512, 512)}


In [3]:
channel_map = json.loads(CHANNEL_MAP_PATH.read_text())
channel_aliases = [entry["alias"] for entry in channel_map]

with tifffile.TiffFile(FULL_MERGE_PATH) as tif:
    series = tif.series[0]
    series_shape = tuple(int(value) for value in series.shape)
    series_axes = series.axes
    level_shapes = [tuple(int(value) for value in level.shape[-2:]) for level in series.levels]
    level_axes = [level.axes for level in series.levels]
    level0_dtype = series.levels[0].dtype
    page0 = series.levels[0].pages[0]
    tile_info = {
        "is_tiled": bool(page0.is_tiled),
        "tilewidth": int(page0.tilewidth) if page0.is_tiled else None,
        "tilelength": int(page0.tilelength) if page0.is_tiled else None,
        "compression": str(page0.compression),
    }

assert series_shape[0] == len(channel_aliases), "Channel map length must match the OME-TIFF channel count"
assert CROP_X >= 0 and CROP_Y >= 0, "Crop origin must be non-negative"
assert CROP_X + CROP_WIDTH <= level_shapes[0][1], "Crop width exceeds image bounds"
assert CROP_Y + CROP_HEIGHT <= level_shapes[0][0], "Crop height exceeds image bounds"

print({
    "series_shape": series_shape,
    "series_axes": series_axes,
    "level_count": len(level_shapes),
    "level0_shape": level_shapes[0],
    "level0_dtype": str(level0_dtype),
    "first_channels": channel_aliases[:5],
    "tile_info": tile_info,
    "crop_shape_yx": (CROP_HEIGHT, CROP_WIDTH),
})


{'series_shape': (25, 62617, 66406), 'series_axes': 'CYX', 'level_count': 8, 'level0_shape': (62617, 66406), 'level0_dtype': 'uint16', 'first_channels': ['R1_DAPI_AF', 'R1_BIT2_RS0584_CY3B', 'R1_BIT3_RS0015_CY5', 'R1_BIT4_RS0083_750', 'R1_DAPI'], 'scale_factors_preview': [{'y': 2, 'x': 2}, {'y': 2, 'x': 2}, {'y': 2, 'x': 2}, {'y': 2, 'x': 2}, {'y': 2, 'x': 2}]}


In [4]:
lazy_image_array = imread(str(FULL_MERGE_PATH))

if lazy_image_array.ndim == 4:
    assert lazy_image_array.shape[0] == 1, "4D images not supported in this barebones test"
    lazy_image_array = da.moveaxis(lazy_image_array[0], 2, 0)
elif lazy_image_array.ndim != 3:
    raise ValueError(f"Unsupported image ndim: {lazy_image_array.ndim}")

lazy_image_array = lazy_image_array[:, CROP_Y : CROP_Y + CROP_HEIGHT, CROP_X : CROP_X + CROP_WIDTH]
lazy_image_array = lazy_image_array.rechunk(CHUNK_SHAPE)

print("crop_box:", {"x": CROP_X, "y": CROP_Y, "width": CROP_WIDTH, "height": CROP_HEIGHT})
print("lazy_image_array:", lazy_image_array)
print("lazy_image_array.shape:", lazy_image_array.shape)
print("lazy_image_array.chunks:", lazy_image_array.chunks)
print("lazy_image_array.chunksize:", lazy_image_array.chunksize)


lazy_image_array: dask.array<rechunk-merge, shape=(25, 62617, 66406), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>
lazy_image_array.shape: (25, 62617, 66406)
lazy_image_array.chunks: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 153), (512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 51

In [5]:
full_image = Image2DModel.parse(
    lazy_image_array,
    dims=("c", "y", "x"),
    c_coords=channel_aliases,
    chunks=CHUNK_SHAPE,
    scale_factors=SCALE_FACTORS,
)

sdata = SpatialData(images={IMAGE_LAYER: full_image})
scale0 = sdata.images[IMAGE_LAYER]["scale0"].image

print({
    "image_layer": IMAGE_LAYER,
    "image_type": type(full_image).__name__,
    "pyramid_levels": list(full_image.keys()),
    "scale0_shape": tuple(scale0.shape),
    "scale0_dims": tuple(scale0.dims),
    "scale0_chunks": scale0.chunks,
})

sdata


{'image_layer': 'full_image', 'image_type': 'DataTree', 'pyramid_levels': ['scale0', 'scale1', 'scale2', 'scale3', 'scale4', 'scale5', 'scale6', 'scale7'], 'scale0_shape': (25, 62617, 66406), 'scale0_dims': ('c', 'y', 'x'), 'scale0_chunks': ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 153), (512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512

SpatialData object
└── Images
      └── 'full_image': DataTree[cyx] (25, 62617, 66406), (25, 31308, 33203), (25, 15654, 16601), (25, 7827, 8300), (25, 3913, 4150), (25, 1956, 2075), (25, 978, 1037), (25, 489, 518)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images)

In [8]:
print({
    "scale0_chunks": full_image.scale0.image.chunks,
    "scale0_shape": tuple(full_image.scale0.image.shape),
    "pyramid_levels": list(full_image.keys()),
})


<xarray.DataArray 'image' (c: 25, y: 62617, x: 66406)> Size: 208GB
dask.array<rechunk-merge, shape=(25, 62617, 66406), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) <U20 2kB 'R1_DAPI_AF' ... 'R4_GFP_POLY_AF488'
  * y        (y) float64 501kB 0.5 1.5 2.5 3.5 ... 6.261e+04 6.262e+04 6.262e+04
  * x        (x) float64 531kB 0.5 1.5 2.5 3.5 ... 6.64e+04 6.64e+04 6.641e+04
Attributes:
    transform:  {'global': Identity }

In [ ]:
if WRITE_PATH.exists():
    shutil.rmtree(WRITE_PATH)

sdata.write(WRITE_PATH, overwrite=True)
print("Wrote:", WRITE_PATH)


/home/ratnayn/miniconda3/envs/spatialdata/lib/python3.12/site-packages/ome_zarr/writer.py:319: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da_delayed = da.to_zarr(


In [ ]:
reloaded = sd.read_zarr(WRITE_PATH)
reloaded_scale0 = reloaded.images[IMAGE_LAYER]["scale0"].image

print(reloaded)
print({
    "images": list(reloaded.images.keys()),
    "labels": list(reloaded.labels.keys()),
    "tables": list(reloaded.tables.keys()),
    "scale0_shape": tuple(reloaded_scale0.shape),
    "scale0_chunks": reloaded_scale0.chunks,
})
